# Módulo 3 · Clase 1: Regresión Lineal — Tu Primer Modelo Predictivo

**Machine Learning for Petroleum Engineers Using Python**  
SLB Ecuador · UDLA · 2026  

Instructor: **Carlos Enrique Mosquera Trujillo**  
Repo: [github.com/cmosquerat/slb-diplomado](https://github.com/cmosquerat/slb-diplomado)

---
## El problema de hoy (y por qué le importa al negocio)

El sensor que mide el **registro sónico** de un pozo se dañó a mitad de la corrida: hay un tramo **sin datos**. Volver a correr la herramienta significa parar el pozo, traer la unidad de registro y gastar **cientos de miles de dólares** — y a veces ni se puede, si el pozo ya está entubado.

**La decisión:** ¿pagamos por re-loguear, o **confiamos en un modelo** que reconstruya el sónico a partir de los sensores que sí funcionaron?

### ¿Qué es Machine Learning?

- **Programación clásica**: el humano escribe la regla (`si presion > 3000: alerta`). Sirve cuando la regla se conoce.
- **Machine Learning**: le damos **ejemplos con la respuesta** y el computador **encuentra la regla** que mejor los explica. Sirve cuando la regla es demasiado compleja para escribirla a mano.
- Hoy: aprendizaje **supervisado** (los ejemplos ya traen la respuesta correcta) y, dentro de él, **regresión** (la respuesta es **un número**: el valor del sónico).

### La gran idea de la clase

1. Las **variables se relacionan**: conocer una te dice algo de la otra (ustedes ya lo usan a diario: más profundidad → más presión).
2. La **correlación** le pone un número a esa relación: dirección (signo) y firmeza (cercanía a ±1).
3. Cuando la relación es **pareja** (lineal), cabe en una fórmula de recta: `y = b + m·x`.
4. Esa fórmula es un **modelo**: una máquina de responder. Entrenarla, evaluarla y usarla = **Machine Learning**.

> 💡 Ejecuta cada celda con `Shift + Enter`. Las celdas 🧩 de práctica están **en blanco**.

---
# 0 · Preparación

In [ ]:
import pandas as pd
import numpy as np

# dataset de registros del pozo 15/9-19 (Volve), ya limpio
REG_URL = "https://raw.githubusercontent.com/cmosquerat/slb-diplomado/main/datos/volve_registros.csv"

reg = pd.read_csv(REG_URL)
print(reg.shape)

---
# 1 · Conocer el dataset (antes de modelar, SIEMPRE)

**Regla de oro del Módulo 2:** nunca modeles datos que no has mirado. Vamos columna por columna.

| Columna | Unidad | Qué mide | Qué nos dice |
|---------|--------|----------|--------------|
| `PROF` | m | profundidad de la medición | identifica la fila |
| `GR`   | GAPI | radioactividad natural | arcilla (alta) vs arena (baja) |
| `DEN`  | g/cm³ | densidad de la roca | qué tan pesada es la roca |
| `NEU`  | % | respuesta al neutrón | porosidad (hidrógeno en los poros) |
| `RDEP` | ohm·m | resistividad profunda | ¿petróleo o agua en los poros? |
| `AC` **(objetivo)** | µs/ft | sónico: lentitud del sonido | roca dura → sonido rápido → AC bajo |

In [ ]:
reg.head()

In [ ]:
reg.tail(3)

### Estadísticas de cada columna

`describe()` nos da el retrato numérico. Leámoslo **con ojos de ingeniero**:

In [ ]:
reg.describe().round(2)

**Lectura física de ese `describe()`:**

- `PROF`: el tramo con datos completos va de ~3 550 a ~4 636 m (la sección profunda del pozo).
- `GR`: media ≈ 36 GAPI → el tramo es **mayormente arenoso** (la arcilla dispara el GR).
- `DEN`: entre 1.9 y 3.0 g/cm³ — rango físicamente razonable para rocas sedimentarias.
- `NEU`: mediana ~15% de porosidad aparente.
- `AC` (el target): va de 40 a 149 µs/ft, media ≈ 80. **Este es el rango que el modelo debe reproducir.**

### La forma de cada variable: histogramas

Un histograma muestra **dónde se concentran** los valores.

In [ ]:
reg["AC"].plot(kind="hist", bins=40, title="Distribucion del sonico (AC)")

In [ ]:
reg["DEN"].plot(kind="hist", bins=40, title="Distribucion de la densidad (DEN)")

### ¿Se relacionan las variables? Primero, dibujarlo

Un punto por profundidad. Si la nube está **inclinada**, hay relación.

In [ ]:
reg.plot(kind="scatter", x="DEN", y="AC", s=2, alpha=0.3,
         title="Sonico vs densidad")

In [ ]:
reg.plot(kind="scatter", x="NEU", y="AC", s=2, alpha=0.3,
         title="Sonico vs neutron")

Ambas nubes están inclinadas: **DEN hacia abajo** (roca más densa → sónico más bajo) y **NEU hacia arriba** (más porosidad → sónico más alto). La física detrás: el sonido viaja rápido en roca compacta y lento en roca porosa.

### Ponerle número: la correlación

La correlación responde **dos preguntas** con un número entre −1 y +1: ¿en qué **dirección** va la relación (signo)? y ¿qué tan **firme** es (cercanía a ±1)?

Regla de bolsillo del curso: |r| > 0.7 fuerte · 0.3–0.7 media · < 0.3 débil.

In [ ]:
reg.corr()["AC"].round(2).sort_values()

- `DEN` = **−0.76**: fuerte y negativa → nuestra mejor candidata.
- `NEU` = +0.72: fuerte y positiva.
- `GR` = +0.36: media. `RDEP` = −0.10: débil, casi no aporta.

> ⚠️ La correlación solo mide relaciones **en línea recta**. Una relación curva clara puede dar un número bajo — lo veremos al final.

---
## 🧩 Práctica 1: Explorar antes de modelar

1. Saca el histograma de `NEU` y el de `GR`. ¿Dónde se concentran?
2. Dibuja el scatter de `GR` vs `AC`. ¿Se ve tan inclinado como el de `DEN`?
3. ¿Cuál es la profundidad **mínima** y **máxima** del dataset? (`min()`, `max()`)
4. Según la tabla de correlaciones, ¿qué feature descartarías de entrada y por qué?

In [ ]:
# Escribe tu solucion aqui


---
# 2 · La mejor recta: regresión lineal simple

La relación DEN–AC es fuerte y aproximadamente recta → la escribimos como fórmula:

$$AC = b + m \times DEN$$

- `m` (**pendiente**) = el *precio del paso*: cuánto cambia AC por cada +1 de DEN.
- `b` (**intercepto**) = el punto de arranque.
- **Entrenar** = encontrar los mejores `m` y `b`. La mejor recta es la que deja los **residuos** (distancias a los puntos) más pequeños.

`scikit-learn` (sklearn) es la librería de ML. Tres pasos: crear → `fit` (entrenar) → `predict`.

In [ ]:
from sklearn.linear_model import LinearRegression

X = reg[["DEN"]]   # features: tabla (corchetes dobles)
y = reg["AC"]      # target: columna

modelo = LinearRegression()
modelo.fit(X, y)   # aqui 'aprende' m y b

print("intercepto b:", modelo.intercept_.round(1))
print("pendiente  m:", modelo.coef_.round(1))

El modelo aprendido: **AC = 280.9 − 82.1 × DEN**.

Lectura: por cada +1 g/cm³ de densidad, el sónico **baja 82 µs/ft**. El signo negativo confirma la física.

### Usarlo: predecir

In [ ]:
# ¿que sonico esperamos donde la densidad es 2.4?
modelo.predict(pd.DataFrame({"DEN": [2.4]}))

### Ver la recta sobre los datos

In [ ]:
ax = reg.plot(kind="scatter", x="DEN", y="AC", s=2, alpha=0.3)

linea = pd.DataFrame({"DEN": [1.9, 3.0]})
linea["AC_pred"] = modelo.predict(linea)
linea.plot(x="DEN", y="AC_pred", color="red", ax=ax,
           title="La recta que mejor se ajusta")

---
# 3 · ¿Qué tan bueno es? Las métricas

- **RMSE / MAE**: el error típico, en las **unidades del target** (µs/ft). Más bajo = mejor.
- **R²**: ¿cuánto mejor que el "modelo tonto" que siempre predice la media? 1 = perfecto, 0 = igual que la media. Se lee como el **porcentaje que el modelo explica**.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

pred = modelo.predict(X)

rmse = mean_squared_error(y, pred) ** 0.5
r2 = r2_score(y, pred)

print("RMSE:", round(rmse, 1), "us/ft")
print("R2:  ", round(r2, 2))

**Lectura:** nos equivocamos ~11.7 µs/ft en promedio, y la densidad sola explica el **57%** del sónico. ¿El otro 43%? Otras variables. Agreguemos más.

### ¿Cuál métrica uso? Criterios y valores aceptables

| Métrica | Cuándo elegirla | Qué es "aceptable" |
|---------|-----------------|---------------------|
| **MAE** | comunicar el error típico en unidades del negocio; robusta a datos raros | júzgala contra la **tolerancia de ingeniería** del uso final |
| **RMSE** | cuando los errores grandes cuestan caro (los castiga más); la estándar para comparar modelos | debe ser **mucho menor** que la desviación estándar del target (si RMSE ≈ desv., el modelo no supera a la media) |
| **R²** | comunicar "% explicado"; comparar entre problemas (no tiene unidades) | > 0.8 excelente · **0.6–0.8 útil en la industria** · 0.4–0.6 débil · < 0.4 insuficiente |

Reglas que no se negocian:

1. Las métricas se reportan **sobre el test**, nunca sobre el train.
2. Compara siempre el RMSE contra el **rango** y la **desviación estándar** del target (aquí: RMSE ~10.6 sobre un rango de 40–149 ≈ 10 % del rango).
3. La decisión final no es estadística: es **¿cuánto cuesta equivocarse** en el uso que le daremos?

---
## 🧩 Práctica 2: Tu propia recta

Repite el flujo completo pero con **NEU** como única feature:

1. Entrena `LinearRegression` con `X = reg[["NEU"]]`.
2. Imprime pendiente e intercepto. ¿El signo de la pendiente tiene sentido físico?
3. Calcula RMSE y R². ¿Es mejor o peor predictor que DEN?

In [ ]:
# Escribe tu solucion aqui


---
# 4 · De una a muchas variables: regresión múltiple

Misma idea, más features: `AC = b + m₁·GR + m₂·DEN + m₃·NEU + m₄·RDEP`. El código casi no cambia.

In [ ]:
features = ["GR", "DEN", "NEU", "RDEP"]
X = reg[features]
y = reg["AC"]

modelo = LinearRegression()
modelo.fit(X, y)

print("R2:", round(modelo.score(X, y), 2))
print(pd.Series(modelo.coef_.round(2), index=features))

R² sube de 0.57 a **0.66**: juntas explican más que la densidad sola.

**Pero cuidado con leer estos coeficientes como "importancia":** DEN = −56 y NEU = 0.4 **no** significa que DEN importe 100 veces más. Los coeficientes dependen de las **unidades**: DEN se mueve entre 2 y 3, NEU entre 0 y 100. Comparar coeficientes de escalas distintas no es justo.

---
# 5 · Estandarizar: poner todo en la misma escala

Cada feature vive en un mundo numérico distinto. Mirémoslo:

In [ ]:
reg[features].plot(kind="box", title="Escalas crudas: incomparables")

**Estandarizar (z-score)**: a cada valor, restarle la media y dividir por la desviación. Después, toda feature tiene media 0 y desviación 1 → **comparables**.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_esc = scaler.fit_transform(X)

modelo_esc = LinearRegression()
modelo_esc.fit(X_esc, y)

importancia = pd.Series(modelo_esc.coef_.round(2), index=features)
importancia.sort_values().plot(kind="barh", title="Ahora si: cual pesa mas")

Con todas en la misma escala: **DEN es la más influyente**, luego NEU; RDEP casi nada — igual que anticipó la correlación. Estandarizar no cambió el R², cambió nuestra **capacidad de interpretar**.

> ⚠️ **Data leakage:** al estandarizar con train/test, el `scaler` se ajusta **solo con el train** (después de separar). Si "ve" el test, la evaluación sale tramposamente optimista.

---
# 6 · Train / Test: ¿sirve con datos que nunca vio?

Hasta ahora evaluamos sobre los **mismos datos** con que entrenamos — como evaluar a un estudiante con el examen que memorizó **con las respuestas**. La prueba honesta: **esconder** una parte de los datos y evaluar solo ahí.

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42)

modelo = LinearRegression()
modelo.fit(X_tr, y_tr)          # entrena con el 75%

print("R2 train:", round(modelo.score(X_tr, y_tr), 3))
print("R2 test: ", round(modelo.score(X_te, y_te), 3))

**Diagnóstico:** R² train ≈ R² test (0.67 vs 0.66) → el modelo **generaliza**, no memorizó.

- Si train ≫ test (p. ej. 0.98 vs 0.55) → **overfitting**: memorizó el ruido.
- Si ambos son bajos → **underfitting**: el modelo es demasiado simple.

### El veredicto visual: predicho vs real (en el test)

In [ ]:
pred_te = modelo.predict(X_te)

comp = pd.DataFrame({"real": y_te, "predicho": pred_te})
ax = comp.plot(kind="scatter", x="real", y="predicho", s=3, alpha=0.3,
               title="Predicho vs real (test)")

# la linea de prediccion perfecta
perfecta = pd.DataFrame({"real": [40, 150], "predicho": [40, 150]})
perfecta.plot(x="real", y="predicho", color="red", ax=ax, legend=False)

---
# 7 · Los límites de la recta

La regresión lineal **asume** que la relación es una recta (el paso siempre vale lo mismo). El mundo real rara vez es tan parejo. Demostración con datos que ya conocen: la **curva de declive** del pozo F-12.

In [ ]:
PROD_URL = "https://raw.githubusercontent.com/cmosquerat/slb-diplomado/main/datos/volve_produccion.csv"
prod = pd.read_csv(PROD_URL)

f12 = prod[(prod["pozo"] == "15/9-F-12") & (prod["oil"] > 0)].copy()
f12 = f12.sort_values("fecha").reset_index(drop=True)
f12["dia"] = range(len(f12))

recta = LinearRegression()
recta.fit(f12[["dia"]], f12["oil"])
f12["oil_recta"] = recta.predict(f12[["dia"]])

ax = f12.plot(kind="scatter", x="dia", y="oil", s=2, alpha=0.3)
f12.plot(x="dia", y="oil_recta", color="red", ax=ax,
         title="Una recta NO describe una curva de declive")

print("R2:", round(recta.score(f12[["dia"]], f12["oil"]), 2))
print("prediccion minima:", round(f12["oil_recta"].min()), "Sm3/dia (!!)")

R² = 0.66 suena decente… pero la recta **predice producción negativa** al final — físicamente imposible. La relación existe y es clara (siempre baja), pero **no es pareja**: cae rápido al inicio y lento después.

**Moraleja:** la recta es el modelo más simple de leer y explicar; cuando el patrón es curvo se necesitan modelos más flexibles — **los veremos en las próximas clases del módulo**.

---
## 🧩 Práctica integradora: El flujo ML completo

Junta todo, de cero, en una sola celda ordenada:

1. **Separar**: train/test (25% test, `random_state=42`) con las 4 features.
2. **Entrenar** `LinearRegression` con el train.
3. **Evaluar** en el test: R², RMSE y MAE (`mean_absolute_error`).
4. **Graficar** predicho vs real del test.
5. **Responder**: ¿usarías este modelo para rellenar el tramo del sensor dañado? ¿Con qué error típico debe contar el geólogo que lo use?

In [ ]:
# Escribe tu solucion aqui


---
## Cierre

**El flujo que aprendieron hoy** (y que se repite en casi todo el módulo):

`datos (X, y)` → `train/test split` → `estandarizar (solo train)` → `entrenar (.fit)` → `evaluar (test)`

**Conceptos:** relación entre variables · correlación (dirección + firmeza) · linealidad (el paso parejo) · modelo = la relación hecha fórmula · RMSE/MAE/R² · estandarización y data leakage · train/test · over/underfitting · los límites de la recta.

**Herramientas:** `LinearRegression` (.fit/.predict/.score) · `train_test_split` · `StandardScaler` · `r2_score`, `mean_squared_error` · `df.corr()`.

---
Carlos Enrique Mosquera Trujillo · cmosquerat@unal.edu.co  
**Machine Learning for Petroleum Engineers Using Python** · SLB Ecuador · UDLA · 2026

*Datos: campo Volve, Equinor (dataset abierto, 2018). Pozo 15/9-19.*